# EXT_AB — BraTS 2024 Post-Treatment: Embedding Extraction + Dice + Master CSV
## External Validation of TaViT V3.2 Pipeline

**Purpose**: Extract nnUNet embeddings for all BraTS-PTG scans, compute segmentation
Dice scores, and build a master CSV compatible with the MU-Glioma TaViT pipeline.

### What This Notebook Does:
1. Load MU-Glioma fine-tuned nnUNet checkpoint (NO re-training)
2. Discover all BraTS-PTG scans (BraTS-GLI-XXXXX-YYY format)
3. Extract 2825-D embeddings per scan (octant + region + vol)
4. **Compute per-scan Dice scores (WT/TC/ET) against expert GT**
5. Build `brats_ptg_master.csv` with zeroed treatment/molecular tokens
6. Create 3-fold stratified CV splits

### Kaggle Datasets Required:
1. `brats-ptg-scans` — BraTS-PTG training_data1_v2/ (scan folders)
2. `nn-unet-mu-glioma` — MU-Glioma nnUNet best checkpoint + **plans.json**
3. `brats-ptg-metadata` — BraTS-PTG supplementary demographic xlsx

### Output:
- `brats_ptg_embeddings.npz` (N, 2825) — scan embeddings
- `brats_ptg_volumes.csv` — tumor volumes per scan
- `brats_ptg_dice_scores.csv` — per-scan Dice scores
- `brats_ptg_master.csv` — full master CSV for TaViT inference
- `brats_ptg_splits.json` — 3-fold CV split assignments

In [1]:
import re
import math
import time
import json
import random
import shutil
import subprocess
import numpy as np
import torch
import torch.nn.functional as F
import gc
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = 'nnunet'
PATCH       = [128, 128, 128]
REGIONS     = ['WT', 'TC', 'ET']
OUTPUT_ROOT = Path('/kaggle/working/ext_brats_ptg')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SYNTHETIC_GAP_DAYS = 90

print(f'Model: {MODEL_NAME} | Patch: {PATCH} | Regions: {REGIONS}')
print(f'Synthetic gap: {SYNTHETIC_GAP_DAYS} days between ordinal timepoints')

Model: nnunet | Patch: [128, 128, 128] | Regions: ['WT', 'TC', 'ET']
Synthetic gap: 90 days between ordinal timepoints


In [2]:
import subprocess, sys, json, time, math, os, shutil
import numpy as np
import torch
import torch.nn.functional as F

try:
    import nnunetv2
    try:
        ver = nnunetv2.__version__
    except AttributeError:
        import importlib.metadata
        try: ver = importlib.metadata.version('nnunetv2')
        except Exception: ver = 'installed'
    print(f'nnunetv2 {ver} ready')
except ImportError:
    print('Installing nnunetv2 ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nnunetv2', '-q'])
    import nnunetv2
    print('nnunetv2 installed')

try:
    import monai
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai[all]', '-q'])
    import monai

import monai.transforms as T
from monai.data import Dataset, DataLoader
from monai.utils import set_determinism
from monai.transforms import MapTransform

set_determinism(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MONAI {monai.__version__} | PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | Total memory: {total_mem:.1f} GB')

usage = shutil.disk_usage('/kaggle/working')
free_gb = usage.free / 1e9
print(f'Disk free: {free_gb:.1f} GB')

Installing nnunetv2 ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


nnunetv2 installed
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
pytensor 2.38.0 requires filelock>=3.15, but you have filelock 3.11.0 which is incompatible.
2026-05-11 03:41:03.099432: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778470863.268360      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778470863.315790      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778470863.710866      22 computation_placer.cc:177] comput

MONAI 1.5.2 | PyTorch 2.10.0+cu128 | Device: cuda
GPU: Tesla T4 | Total memory: 15.6 GB
Disk free: 20.9 GB


In [3]:
# Label mapping (IDENTICAL to MU-Glioma / BraTS 2024)
class ConvertToMultiChannelBrats3Chd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img==1)|(img==2)|(img==3),  # WT = NETC+SNFH+ET (no RC)
                (img==1)|(img==3),           # TC = NETC+ET
                img==3,                      # ET = Enhancing Tissue
            ]
            d[key] = (torch.stack(result, 0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, 0).astype(np.float32))
        return d

print('Label mapping: WT=1+2+3 | TC=1+3 | ET=3 (identical to MU-Glioma)')

Label mapping: WT=1+2+3 | TC=1+3 | ET=3 (identical to MU-Glioma)


In [4]:
# ═══ CELL 4: BraTS-PTG Scan Discovery ═══
import nibabel as nib

SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def setup_nii_gz_symlinks(data_dir):
    count = 0
    for nii_gz in Path(data_dir).rglob('*.nii_gz'):
        real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
        link = SYMLINK_DIR / nii_gz.parent.name / real_name
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            os.symlink(str(nii_gz), str(link))
            count += 1
    return count

DATA_ROOT = Path('/kaggle/input')
for ds_dir in DATA_ROOT.iterdir():
    if not ds_dir.is_dir(): continue
    nii_gz_files = list(ds_dir.rglob('*.nii_gz'))
    if nii_gz_files:
        n = setup_nii_gz_symlinks(ds_dir)
        if n: print(f'  Created {n} symlinks from {ds_dir.name}')

NIFTI_ROOT = None
for search_root in [SYMLINK_DIR, DATA_ROOT]:
    if not search_root.exists(): continue
    for c in search_root.rglob('BraTS-GLI-*'):
        if c.is_dir():
            if list(c.glob('*-t1c*')):
                NIFTI_ROOT = c.parent
                break
    if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    raise RuntimeError('No BraTS-GLI-* folders found')
print(f'NIFTI_ROOT: {NIFTI_ROOT}')

all_scans = []
for d in sorted(NIFTI_ROOT.iterdir()):
    if not d.is_dir() or 'BraTS-GLI' not in d.name: continue
    parts = d.name.rsplit('-', 1)
    if len(parts) != 2: continue
    pid, tp = parts[0], parts[1]
    files = {m: list(d.glob(f'*-{m}*')) for m in ['t1n', 't1c', 't2w', 't2f']}
    seg = list(d.glob('*-seg*'))
    if not (all(files[m] for m in files) and seg): continue
    all_scans.append({
        'scan_id': d.name, 'patient_id': pid, 'timepoint': tp,
        't1n': str(files['t1n'][0]), 't1c': str(files['t1c'][0]),
        't2w': str(files['t2w'][0]), 't2f': str(files['t2f'][0]),
        'seg': str(seg[0]),
    })

from collections import Counter
pid_counts = Counter(s['patient_id'] for s in all_scans)
print(f'Total scans: {len(all_scans)} | Patients: {len(pid_counts)}')
print(f'≥2 tp: {sum(1 for c in pid_counts.values() if c>=2)} | ≥3 tp: {sum(1 for c in pid_counts.values() if c>=3)}')

if all_scans:
    s = all_scans[0]
    print(f'Spot-check: {s["scan_id"]} → nibabel shape: {nib.load(s["t1c"]).shape}')

  Created 8105 symlinks from datasets
NIFTI_ROOT: /kaggle/working/nifti_links
Total scans: 1621 | Patients: 731
≥2 tp: 559 | ≥3 tp: 153
Spot-check: BraTS-GLI-00005-100 → nibabel shape: (182, 218, 182)


In [5]:
patch = [128, 128, 128]
val_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats3Chd(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
print('Val transforms ready')

Val transforms ready


In [6]:
def validate_scan(s):
    try:
        for key in ['t1n','t1c','t2w','t2f','seg']:
            p = Path(s[key])
            if not p.exists(): return False
            if p.stat().st_size < 1024: return False
        return True
    except Exception:
        return False

def build_dicts(scan_list):
    dicts, bad = [], []
    for s in scan_list:
        if not validate_scan(s):
            bad.append(s['scan_id']); continue
        dicts.append({
            'image': [s['t1n'],s['t1c'],s['t2w'],s['t2f']],
            'label': s['seg'],
            'patient_id': s['patient_id'],
            'timepoint':  s['timepoint'],
        })
    if bad: print(f'  Skipped {len(bad)} invalid: {bad[:5]}')
    return dicts

print('Validating scans...')
all_dicts = build_dicts(all_scans)
print(f'Valid: {len(all_dicts)} / {len(all_scans)}')

Validating scans...
Valid: 1621 / 1621


In [7]:
# ═══ CELL 7: Load nnUNet (PlainConvUNet from plans.json) ═══
print('='*55)
print('  Loading nnUNet — MU-Glioma fine-tuned checkpoint')
print('  REQUIRES: plans.json alongside checkpoint!')
print('='*55)

ckpt_path  = None
plans_path = None
for name in ['nnunet_best.pth', 'checkpoint_final.pth']:
    for p in Path('/kaggle/input').rglob(name):
        ckpt_path = p; break
    if ckpt_path: break
for fname in ['plans.json', 'nnUNetPlans.json']:
    for p in Path('/kaggle/input').rglob(fname):
        plans_path = p; break
    if plans_path: break

print(f'Checkpoint: {ckpt_path}')
print(f'Plans:      {plans_path}')

if plans_path is None:
    raise RuntimeError(
        '\n\n'
        '╔══════════════════════════════════════════════════════════╗\n'
        '║  CRITICAL: plans.json NOT FOUND!                       ║\n'
        '║                                                        ║\n'
        '║  Without plans.json, the code falls back to DynUNet    ║\n'
        '║  which produces 42-D embeddings instead of 2825-D.     ║\n'
        '║                                                        ║\n'
        '║  FIX: Upload plans.json alongside nnunet_best.pth      ║\n'
        '║  in your Kaggle dataset.                               ║\n'
        '║                                                        ║\n'
        '║  File location on your machine:                        ║\n'
        '║  implementation_mu_glioma/Phase_M2/outputs/plans.json  ║\n'
        '╚══════════════════════════════════════════════════════════╝\n'
    )

model = None

def safe_torch_load(path):
    import numpy
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        pass
    try:
        safe = [numpy.core.multiarray.scalar, numpy.dtype, numpy.ndarray]
        with torch.serialization.safe_globals(safe):
            return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        pass
    return torch.load(path, map_location='cpu', weights_only=False)

try:
    import json as _j, torch.nn as nn
    plans = _j.load(open(plans_path))
    cfg   = plans['configurations']['3d_fullres']
    n_stages = len(cfg['conv_kernel_sizes'])
    base_f   = cfg.get('UNet_base_num_features', 32)
    max_f    = cfg.get('unet_max_num_features', 320)
    features = [min(base_f * (2**i), max_f) for i in range(n_stages)]
    strides  = cfg['pool_op_kernel_sizes']
    kernels  = cfg['conv_kernel_sizes']
    n_enc    = cfg.get('n_conv_per_stage_encoder', [2]*n_stages)
    n_dec    = cfg.get('n_conv_per_stage_decoder', [2]*(n_stages-1))
    print(f'  PlainConvUNet | {n_stages} stages | features: {features}')

    PlainConvUNet = None
    for imp in [
        ('dynamic_network_architectures.architectures.unet', 'PlainConvUNet'),
        ('nnunetv2.architectures.neural_network',            'PlainConvUNet'),
        ('nnunetv2.nets.UNet',                               'PlainConvUNet'),
    ]:
        try:
            mod = __import__(imp[0], fromlist=[imp[1]])
            PlainConvUNet = getattr(mod, imp[1])
            print(f'  Imported from {imp[0]}')
            break
        except Exception:
            continue

    if PlainConvUNet is None:
        raise ImportError('Could not import PlainConvUNet')

    model = PlainConvUNet(
        input_channels          = 4,
        n_stages                = n_stages,
        features_per_stage      = features,
        conv_op                 = nn.Conv3d,
        kernel_sizes            = kernels,
        strides                 = strides,
        n_conv_per_stage        = n_enc,
        num_classes             = 3,
        n_conv_per_stage_decoder= n_dec,
        conv_bias               = False,
        norm_op                 = nn.InstanceNorm3d,
        norm_op_kwargs          = {'eps': 1e-05, 'affine': True},
        dropout_op              = None,
        dropout_op_kwargs       = None,
        nonlin                  = nn.LeakyReLU,
        nonlin_kwargs           = {'inplace': True},
        deep_supervision        = False,
    )

    ckpt = safe_torch_load(ckpt_path)
    # Print training metrics stored in checkpoint
    ckpt_epoch = ckpt.get('epoch', '?')
    ckpt_dice  = ckpt.get('best_dice', None)
    if ckpt_dice is not None:
        print(f'  Checkpoint from epoch {ckpt_epoch} | Training best Dice: {ckpt_dice:.4f}')
    else:
        print(f'  Checkpoint from epoch {ckpt_epoch} (no Dice stored)')
    # Load per-region metrics if saved
    ckpt_metrics = ckpt.get('metrics', {})
    if ckpt_metrics and 'per_region' in ckpt_metrics:
        last_pr = ckpt_metrics['per_region'][-1] if ckpt_metrics['per_region'] else None
        if last_pr:
            print(f'  Training per-region: WT={last_pr[0]:.3f} TC={last_pr[1]:.3f} ET={last_pr[2]:.3f}')
    state = ckpt.get('network_weights', ckpt.get('model', ckpt.get('state_dict', ckpt)))
    own   = model.state_dict()
    compat = {k: v for k,v in state.items() if k in own and own[k].shape == v.shape}
    model.load_state_dict({**own, **compat}, strict=False)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Loaded: {len(compat)}/{len(own)} layers | {n_params/1e6:.1f}M params')
    if len(compat) < len(own) // 2:
        print('  ⚠️ WARNING: <50% layers matched — check checkpoint compatibility!')
except Exception as e:
    raise RuntimeError(f'PlainConvUNet build failed: {e}. Upload plans.json!')

model = model.to(device)
model.eval()
print(f'\n✅ PlainConvUNet on {device} — INFERENCE MODE')
print(f'   Stage 3 features: {features[3]} channels (embeddings will be {8*features[3] + 3*features[3] + 9}-D)')

  Loading nnUNet — MU-Glioma fine-tuned checkpoint
  REQUIRES: plans.json alongside checkpoint!
Checkpoint: /kaggle/input/datasets/zinou123viva/nn-unet-pretrained-brats2024/nnunet_best.pth
Plans:      /kaggle/input/datasets/zinou123viva/nn-unet-pretrained-brats2024/plans.json
  PlainConvUNet | 6 stages | features: [32, 64, 128, 256, 320, 320]
  Imported from dynamic_network_architectures.architectures.unet
  Checkpoint from epoch 27 | Training best Dice: 0.8150
  Loaded: 219/219 layers | 30.8M params

✅ PlainConvUNet on cuda — INFERENCE MODE
   Stage 3 features: 256 channels (embeddings will be 2825-D)


In [8]:
# ═══════════════════════════════════════════════════════════════
#  CELL 8: Embedding Extraction + Dice Score Evaluation
#  Output: octant(8×256) + region(3×256) + vol(9) = 2825-D
#  Also computes per-scan Dice scores for WT/TC/ET
# ═══════════════════════════════════════════════════════════════

def _is_corrupt_file_error(exc):
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream',
            'corrupt','truncat','LoadImaged','applying transform']
    e = exc
    while e is not None:
        if any(k in (type(e).__name__+' '+str(e)) for k in SKIP): return True
        e = e.__cause__ or e.__context__
    return False

def safe_emb_iter(loader):
    it, n_skip = iter(loader), 0
    while True:
        try: yield next(it)
        except StopIteration:
            if n_skip: print(f'  Skipped {n_skip} corrupt files')
            return
        except Exception as e:
            if _is_corrupt_file_error(e): n_skip += 1; continue
            raise

def compute_dice(pred, gt, eps=1e-7):
    """Compute Dice score. Returns NaN if GT region is empty."""
    gt_sum = gt.sum().item()
    if gt_sum < 1:
        return float('nan')
    intersection = (pred * gt).sum().item()
    return (2.0 * intersection + eps) / (pred.sum().item() + gt_sum + eps)

def get_wt_bbox(lbl_3ch, min_size=2):
    wt = lbl_3ch[0]
    coords = (wt > 0.5).nonzero(as_tuple=True)
    if len(coords[0]) < min_size: return None
    z0, z1 = int(coords[0].min()), int(coords[0].max()) + 1
    y0, y1 = int(coords[1].min()), int(coords[1].max()) + 1
    x0, x1 = int(coords[2].min()), int(coords[2].max()) + 1
    D, H, W = wt.shape
    z0, z1 = max(z0-1, 0), min(z1+1, D)
    y0, y1 = max(y0-1, 0), min(y1+1, H)
    x0, x1 = max(x0-1, 0), min(x1+1, W)
    if z1-z0 < 2: z1 = min(z0+2, D)
    if y1-y0 < 2: y1 = min(y0+2, H)
    if x1-x0 < 2: x1 = min(x0+2, W)
    return (z0, z1, y0, y1, x0, x1)

def extract_embeddings(model):
    model.eval()
    _feats = {}; hooks = []

    # Hook encoder stage 3 (should be 256ch at 16³)
    if hasattr(model, 'encoder') and hasattr(model.encoder, 'stages'):
        stages = list(model.encoder.stages)
        target_idx = min(3, len(stages) - 1)
        def _hook(m, inp, out):
            feat = out[-1] if isinstance(out, (list, tuple)) else out
            _feats['mid'] = feat.detach()
        hooks.append(stages[target_idx].register_forward_hook(_hook))
        print(f'  Hook: encoder.stages[{target_idx}]')
    else:
        raise RuntimeError('Model has no encoder.stages — NOT PlainConvUNet. Upload plans.json!')

    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(exist_ok=True)
    embs, ids, tps = [], [], []
    dice_scores = {'WT': [], 'TC': [], 'ET': []}  # Dice accumulators

    ds     = Dataset(all_dicts, val_transforms)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    total  = len(all_dicts)
    n_skip = 0; n_empty = 0
    t_start = time.time()

    print(f'  Extracting embeddings + Dice: {total} scans')
    print(f'  {"─"*65}')

    with torch.no_grad():
        for idx, batch in enumerate(safe_emb_iter(loader)):
            pid = batch['patient_id'][0]
            tp  = batch['timepoint'][0]
            try:
                img = batch['image'].to(device)
                lbl = batch['label'].to(device)
                img_p = F.interpolate(img, PATCH, mode='trilinear', align_corners=False)
                lbl_p = F.interpolate(lbl.float(), PATCH, mode='nearest')
                _feats.clear()
                seg_logits = model(img_p)  # (1, 3, 128, 128, 128)

                # ── Dice Score (free — we already ran inference) ──
                seg_pred = (torch.sigmoid(seg_logits) > 0.5).float()
                for ch, reg in enumerate(['WT', 'TC', 'ET']):
                    d = compute_dice(seg_pred[0, ch], lbl_p[0, ch])
                    dice_scores[reg].append(d)

                if 'mid' not in _feats:
                    n_skip += 1; continue

                feat = _feats['mid']
                C = feat.shape[1]

                if idx == 0:
                    print(f'  Feature map: {tuple(feat.shape)} → C={C}')
                    if C < 64:
                        raise RuntimeError(
                            f'CRITICAL: C={C} — expected 256. '
                            f'Model is NOT PlainConvUNet. STOP and upload plans.json!')

                wt_vol = float(lbl_p[0, 0].sum().item())
                tc_vol = float(lbl_p[0, 1].sum().item())
                et_vol = float(lbl_p[0, 2].sum().item())

                lbl_feat = F.adaptive_avg_pool3d(lbl_p, feat.shape[2:])

                # Octant Spatial Pooling (8 × C)
                feat_crop = None
                bbox = get_wt_bbox(lbl_feat[0], min_size=2)

                if bbox is not None:
                    z0, z1, y0, y1, x0, x1 = bbox
                    feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]
                    oct_pooled = F.adaptive_avg_pool3d(feat_crop, (2, 2, 2))
                    oct_vec = oct_pooled[0].reshape(C, 8).T.reshape(-1)
                    lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]
                    feat_flat = feat_crop[0].reshape(C, -1)
                    region_vecs = []
                    for ch in range(3):
                        mask = lbl_crop[0, ch].reshape(-1)
                        if float(mask.sum().item()) > 0.01:
                            rvec = (feat_flat * mask.unsqueeze(0)).sum(1) / mask.sum()
                        else:
                            rvec = torch.zeros(C, device=device)
                        region_vecs.append(rvec)
                else:
                    oct_vec = torch.zeros(8 * C, device=device)
                    region_vecs = [torch.zeros(C, device=device) for _ in range(3)]
                    n_empty += 1

                # Volumetric morphology (9-D)
                log_wt = np.log1p(wt_vol)
                log_tc = np.log1p(tc_vol)
                log_et = np.log1p(et_vol)
                has_wt = 1.0 if wt_vol > 10 else 0.0
                has_tc = 1.0 if tc_vol > 10 else 0.0
                has_et = 1.0 if et_vol > 10 else 0.0
                tc_wt = tc_vol / (wt_vol + 1e-6)
                et_wt = et_vol / (wt_vol + 1e-6)
                et_tc = et_vol / (tc_vol + 1e-6)
                vol_feat = torch.tensor(
                    [log_wt, log_tc, log_et, has_wt, has_tc, has_et,
                     tc_wt, et_wt, et_tc], dtype=torch.float32)

                # Concatenate: [octant(8C) + region(3C) + vol(9)]
                emb = torch.cat([oct_vec.cpu()] + [v.cpu() for v in region_vecs]
                                + [vol_feat]).numpy()

                del img, lbl, img_p, lbl_p, feat, seg_logits, seg_pred
                if feat_crop is not None: del feat_crop

                embs.append(emb); ids.append(pid); tps.append(tp)

                if idx == 0:
                    print(f'  Embedding dim: {emb.shape[0]}-D (8×{C} + 3×{C} + 9)')
                    assert emb.shape[0] == 2825, f'Expected 2825-D, got {emb.shape[0]}'
                    print(f'  ✅ Dimension verified: 2825-D')
                    print(f'  {"─"*65}')

                if (idx+1) % 100 == 0 or idx == 0:
                    elapsed = time.time() - t_start
                    rate = (idx+1) / max(elapsed, 1e-6)
                    eta = (total - idx - 1) / max(rate, 1e-6)
                    wt_d = np.nanmean(dice_scores['WT']) if dice_scores['WT'] else 0
                    tc_d = np.nanmean(dice_scores['TC']) if dice_scores['TC'] else 0
                    et_d = np.nanmean(dice_scores['ET']) if dice_scores['ET'] else 0
                    print(f'  [{idx+1:5d}/{total}] {pid} tp={tp} '
                          f'Dice(WT={wt_d:.3f} TC={tc_d:.3f} ET={et_d:.3f}) '
                          f'| {rate:.1f}/s ETA {eta/60:.1f}m')

            except Exception as e:
                print(f'  [{idx+1:5d}/{total}] ERROR {pid}: {str(e)[:80]}')
                n_skip += 1; continue

    for hk in hooks:
        try: hk.remove()
        except: pass

    # ── Dice Summary: MU-Glioma vs BraTS-PTG ──
    # Read training Dice from checkpoint (if available)
    ckpt_data = safe_torch_load(ckpt_path) if ckpt_path else {}
    ckpt_dice_mean = ckpt_data.get('best_dice', 0.877)
    ckpt_epoch = ckpt_data.get('epoch', '?')
    ckpt_metrics = ckpt_data.get('metrics', {})
    if ckpt_metrics and 'per_region' in ckpt_metrics and ckpt_metrics['per_region']:
        last_pr = ckpt_metrics['per_region'][-1]
        muglioma = {'WT': last_pr[0], 'TC': last_pr[1], 'ET': last_pr[2]}
        print(f'  MU-Glioma baseline (from checkpoint ep {ckpt_epoch}): WT={last_pr[0]:.3f} TC={last_pr[1]:.3f} ET={last_pr[2]:.3f}')
    else:
        # Fallback: use best_dice as mean, estimate per-region from training log
        muglioma = {'WT': 0.904, 'TC': 0.857, 'ET': 0.872}
        print(f'  MU-Glioma baseline (from training log): WT=0.904 TC=0.857 ET=0.872 (mean={ckpt_dice_mean:.4f})')
    del ckpt_data
    print(f'  {"═"*65}')
    print(f'  nnUNet Dice — MU-Glioma (train) vs BraTS-PTG (zero-shot)')
    print(f'  {"═"*65}')
    print(f'  {"Region":<8} {"MU-Glioma":>12} {"BraTS-PTG":>16} {"Drop":>10}')
    print(f'  {"─"*50}')
    for reg in ['WT', 'TC', 'ET']:
        vals = [v for v in dice_scores[reg] if not np.isnan(v)]
        if vals:
            ext_m = np.mean(vals)
            drop = (muglioma[reg] - ext_m) / muglioma[reg] * 100
            print(f'  {reg:<8} {muglioma[reg]:>12.3f} {ext_m:>10.3f}±{np.std(vals):.3f} {drop:>+9.1f}%')
    mean_mu = np.mean(list(muglioma.values()))
    mean_ext = np.nanmean([np.nanmean(dice_scores[r]) for r in ['WT','TC','ET']])
    drop_mean = (mean_mu - mean_ext) / mean_mu * 100
    print(f'  {"─"*50}')
    print(f'  {"Mean":<8} {mean_mu:>12.3f} {mean_ext:>16.3f} {drop_mean:>+9.1f}%')
    print(f'  {"═"*65}')
    print(f'  Embeddings: {len(embs)}/{total} in {(time.time()-t_start)/60:.1f}m')
    print(f'  Skipped: {n_skip} | Empty ROI: {n_empty}')

    if not embs: raise RuntimeError('No embeddings extracted.')
    arr = np.array(embs)
    out = emb_dir / 'brats_ptg_embeddings.npz'
    np.savez_compressed(out, embeddings=arr,
                        patient_ids=np.array(ids), timepoints=np.array(tps))
    print(f'  Saved: {out}  shape={arr.shape}')

    # Save volumes CSV
    import pandas as pd
    vol_rows = []
    for j, (pid, tp, emb) in enumerate(zip(ids, tps, embs)):
        v = emb[-9:]
        vol_rows.append({
            'patient_id': pid, 'timepoint': tp,
            'wt_vol': float(np.expm1(v[0])),
            'tc_vol': float(np.expm1(v[1])),
            'et_vol': float(np.expm1(v[2])),
            'has_wt': float(v[3]), 'has_tc': float(v[4]), 'has_et': float(v[5]),
            'tc_wt_ratio': float(v[6]), 'et_wt_ratio': float(v[7]),
            'et_tc_ratio': float(v[8]),
        })
    vol_df = pd.DataFrame(vol_rows)
    csv_out = emb_dir / 'brats_ptg_volumes.csv'
    vol_df.to_csv(csv_out, index=False)
    print(f'  Volumes: {csv_out}')

    # Save Dice scores CSV
    dice_rows = []
    for j in range(len(dice_scores['WT'])):
        dice_rows.append({
            'patient_id': ids[j] if j < len(ids) else 'unknown',
            'timepoint': tps[j] if j < len(tps) else 'unknown',
            'dice_wt': dice_scores['WT'][j],
            'dice_tc': dice_scores['TC'][j],
            'dice_et': dice_scores['ET'][j],
        })
    dice_df = pd.DataFrame(dice_rows)
    dice_out = emb_dir / 'brats_ptg_dice_scores.csv'
    dice_df.to_csv(dice_out, index=False)
    print(f'  Dice scores: {dice_out}')

    return arr

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# Skip extraction if embeddings already exist (saves ~57 min on re-run)
# Check Kaggle input dataset first, then working directory
emb_cache = None
for _p in [
    Path('/kaggle/input/datasets/zinou123viva/embading-brats-glioma-mupretrained/brats_ptg_embeddings.npz'),
    Path('/kaggle/input') / 'embading-brats-glioma-mupretrained' / 'brats_ptg_embeddings.npz',
    OUTPUT_ROOT / 'embeddings' / 'brats_ptg_embeddings.npz',
]:
    if _p.exists():
        emb_cache = _p; break
# Also find cached dice/volumes alongside embeddings
if emb_cache is not None:
    _emb_dir = emb_cache.parent
    for _csv in ['brats_ptg_dice_scores.csv', 'brats_ptg_volumes.csv']:
        _src = _emb_dir / _csv
        _dst = OUTPUT_ROOT / 'embeddings' / _csv
        if _src.exists() and not _dst.exists():
            _dst.parent.mkdir(parents=True, exist_ok=True)
            import shutil; shutil.copy2(_src, _dst)
            print(f'  Copied {_csv} → {_dst}')
    # Also copy embeddings to working dir for downstream cells
    _dst_emb = OUTPUT_ROOT / 'embeddings' / 'brats_ptg_embeddings.npz'
    if not _dst_emb.exists():
        _dst_emb.parent.mkdir(parents=True, exist_ok=True)
        import shutil; shutil.copy2(emb_cache, _dst_emb)
        print(f'  Copied embeddings → {_dst_emb}')

if emb_cache is not None:
    cached = np.load(emb_cache, allow_pickle=True)
    embeddings = cached['embeddings']
    print(f'\n✅ CACHED embeddings loaded: {emb_cache}')
    print(f'   Shape: {embeddings.shape} — skipping extraction (already done)')
    # Also print cached Dice if available
    dice_cache = OUTPUT_ROOT / 'embeddings' / 'brats_ptg_dice_scores.csv'
    if dice_cache.exists():
        import pandas as pd
        ddf = pd.read_csv(dice_cache)
        print(f'   Cached Dice: WT={ddf["dice_wt"].mean():.4f}  TC={ddf["dice_tc"].mean():.4f}  ET={ddf["dice_et"].mean():.4f}')
else:
    print('\nExtracting nnUNet embeddings + Dice for BraTS-PTG...')
    embeddings = extract_embeddings(model)


Extracting nnUNet embeddings + Dice for BraTS-PTG...
  Hook: encoder.stages[3]
  Extracting embeddings + Dice: 1621 scans
  ─────────────────────────────────────────────────────────────────
  Feature map: (1, 256, 16, 16, 16) → C=256
  Embedding dim: 2825-D (8×256 + 3×256 + 9)
  ✅ Dimension verified: 2825-D
  ─────────────────────────────────────────────────────────────────
  [    1/1621] BraTS-GLI-00005 tp=100 Dice(WT=0.856 TC=nan ET=nan) | 0.4/s ETA 64.7m
  [  100/1621] BraTS-GLI-02064 tp=102 Dice(WT=0.851 TC=0.728 ET=0.717) | 0.6/s ETA 43.0m
  [  200/1621] BraTS-GLI-02102 tp=100 Dice(WT=0.875 TC=0.732 ET=0.715) | 0.5/s ETA 48.0m
  [  300/1621] BraTS-GLI-02144 tp=100 Dice(WT=0.879 TC=0.732 ET=0.715) | 0.5/s ETA 46.6m
  [  400/1621] BraTS-GLI-02193 tp=104 Dice(WT=0.880 TC=0.732 ET=0.719) | 0.5/s ETA 43.8m
  [  500/1621] BraTS-GLI-02230 tp=101 Dice(WT=0.874 TC=0.708 ET=0.706) | 0.5/s ETA 38.1m
  [  600/1621] BraTS-GLI-02307 tp=100 Dice(WT=0.873 TC=0.701 ET=0.700) | 0.5/s ETA 34.4m
  [

In [9]:
# ═══ CELL 9: Build Master CSV ═══
import pandas as pd

print('='*55)
print('  Building brats_ptg_master.csv')
print('  Treatment: ZEROS | Molecular: ZEROS')
print('='*55)

emb_dir = OUTPUT_ROOT / 'embeddings'
emb_dir.mkdir(parents=True, exist_ok=True)
vol_path = emb_dir / 'brats_ptg_volumes.csv'

# Search: local working dir → Kaggle input datasets → reconstruct from embeddings
vols = None
for _vp in [
    vol_path,
    Path('/kaggle/input/datasets/zinou123viva/brats2024-metadata/tumor_volumes.csv'),
    Path('/kaggle/input') / 'brats2024-metadata' / 'tumor_volumes.csv',
    Path('/kaggle/input/datasets/zinou123viva/embading-brats-glioma-mupretrained/brats_ptg_volumes.csv'),
]:
    if _vp.exists():
        vols = pd.read_csv(_vp)
        print(f'Loaded volumes from: {_vp} ({len(vols)} rows)')
        # Ensure expected columns exist — rename if needed
        col_map = {}
        for c in vols.columns:
            cl = c.lower()
            if 'patient' in cl and 'id' in cl: col_map[c] = 'patient_id'
            if 'wt' in cl and 'vol' in cl: col_map[c] = 'wt_vol'
            if 'tc' in cl and 'vol' in cl: col_map[c] = 'tc_vol'
            if 'et' in cl and 'vol' in cl: col_map[c] = 'et_vol'
        if col_map:
            vols = vols.rename(columns=col_map)
        break

if vols is None:
    # Final fallback: reconstruct from embeddings
    print('  No volumes CSV found — reconstructing from embeddings...')
    emb_data = np.load(emb_dir / 'brats_ptg_embeddings.npz', allow_pickle=True)
    embs = emb_data['embeddings']
    pids = emb_data['patient_ids']
    tpts = emb_data['timepoints']
    vol_rows = []
    for j in range(len(pids)):
        v = embs[j, -9:]
        vol_rows.append({
            'patient_id': str(pids[j]), 'timepoint': str(tpts[j]),
            'wt_vol': float(np.expm1(v[0])),
            'tc_vol': float(np.expm1(v[1])),
            'et_vol': float(np.expm1(v[2])),
            'has_wt': float(v[3]), 'has_tc': float(v[4]), 'has_et': float(v[5]),
            'tc_wt_ratio': float(v[6]), 'et_wt_ratio': float(v[7]),
            'et_tc_ratio': float(v[8]),
        })
    vols = pd.DataFrame(vol_rows)
    print(f'  Reconstructed: {len(vols)} scans')

# Save to working dir for downstream
if not vol_path.exists():
    vols.to_csv(vol_path, index=False)
# Compute derived columns if missing
if 'has_wt' not in vols.columns:
    vols['has_wt'] = (vols['wt_vol'] > 10).astype(float)
    vols['has_tc'] = (vols['tc_vol'] > 10).astype(float)
    vols['has_et'] = (vols['et_vol'] > 10).astype(float)
    vols['tc_wt_ratio'] = vols['tc_vol'] / (vols['wt_vol'] + 1e-6)
    vols['et_wt_ratio'] = vols['et_vol'] / (vols['wt_vol'] + 1e-6)
    vols['et_tc_ratio'] = vols['et_vol'] / (vols['tc_vol'] + 1e-6)
    print(f'  Computed derived columns (has_wt/tc/et, ratios)')
print(f'Volumes ready: {len(vols)} scans')

meta = None
for p in Path('/kaggle/input').rglob('*.xlsx'):
    if 'PTG' in p.name or 'demographic' in p.name.lower() or 'metadata' in p.name.lower():
        try:
            import openpyxl
            meta = pd.read_excel(p)
            print(f'Loaded metadata from: {p.name}')
        except:
            pass
        break

master = vols.copy()
master['scan_id'] = master['patient_id'] + '-' + master['timepoint'].astype(str)
master['tp_ordinal'] = pd.to_numeric(master['timepoint'], errors='coerce').fillna(100).astype(int)
master['days_from_diagnosis'] = (master['tp_ordinal'] - 100) * SYNTHETIC_GAP_DAYS
master['days_between_scans'] = SYNTHETIC_GAP_DAYS

if meta is not None:
    meta_cols = meta.columns.tolist()
    id_col = [c for c in meta_cols if 'Subject' in c or 'ID' in c]
    if id_col:
        meta['pid'] = meta[id_col[0]].str.rsplit('-', n=1).str[0]
        meta_dedup = meta.drop_duplicates('pid')
        age_col = [c for c in meta_cols if 'Age' in c]
        sex_col = [c for c in meta_cols if 'Sex' in c]
        glioma_col = [c for c in meta_cols if 'Glioma' in c or 'glioma' in c]
        if age_col:
            master['age_at_diagnosis'] = master['patient_id'].map(dict(zip(meta_dedup['pid'], meta_dedup[age_col[0]]))).fillna(55)
        else:
            master['age_at_diagnosis'] = 55
        if sex_col:
            master['sex'] = master['patient_id'].map(dict(zip(meta_dedup['pid'], meta_dedup[sex_col[0]]))).fillna('Unknown')
        else:
            master['sex'] = 'Unknown'
        if glioma_col:
            master['glioma_type'] = master['patient_id'].map(dict(zip(meta_dedup['pid'], meta_dedup[glioma_col[0]]))).fillna('Unknown')
        else:
            master['glioma_type'] = 'Unknown'
        print(f'  Demographics merged: {len(meta_dedup)} patients')
else:
    master['age_at_diagnosis'] = 55
    master['sex'] = 'Unknown'
    master['glioma_type'] = 'Unknown'
    print('  No metadata — using defaults')

tv_cols = ['tv_chemo_act','tv_radio_act','tv_avastin_act','tv_maint_act',
           'tv_post_chemo','tv_post_radio','tv_rt_dose_n','tv_n_surg_n']
for col in tv_cols:
    master[col] = 0.0
for i in range(26):
    master[f'mol_{i:02d}'] = 0.0
print(f'  Treatment (8-D): ZEROS | Molecular (26-D): ZEROS')

def classify_trajectory(group):
    g = group.sort_values('tp_ordinal')
    if len(g) < 2:
        return pd.Series(['STABLE'] * len(g), index=g.index)
    first_wt, last_wt = g.iloc[0]['wt_vol'], g.iloc[-1]['wt_vol']
    if first_wt < 10:
        label = 'STABLE'
    else:
        ratio = (last_wt - first_wt) / (first_wt + 1e-6)
        label = 'PROG' if ratio > 0.2 else ('RESP' if ratio < -0.2 else 'STABLE')
    return pd.Series([label] * len(g), index=g.index)

master['traj_class_v2'] = master.groupby('patient_id', group_keys=False).apply(classify_trajectory)
traj_dist = master.drop_duplicates('patient_id')['traj_class_v2'].value_counts()
print(f'\nTrajectory distribution:')
for cls, n in traj_dist.items():
    print(f'  {cls}: {n}')

master_path = OUTPUT_ROOT / 'brats_ptg_master.csv'
master.to_csv(master_path, index=False)
print(f'\nSaved: {master_path} ({master.shape})')

  Building brats_ptg_master.csv
  Treatment: ZEROS | Molecular: ZEROS
Loaded volumes from: /kaggle/working/ext_brats_ptg/embeddings/brats_ptg_volumes.csv (1620 rows)
Volumes ready: 1620 scans
Loaded metadata from: BraTS-PTG supplementary demographic information and metadata.xlsx
  Demographics merged: 818 patients
  Treatment (8-D): ZEROS | Molecular (26-D): ZEROS

Trajectory distribution:
  STABLE: 420
  PROG: 231
  RESP: 80

Saved: /kaggle/working/ext_brats_ptg/brats_ptg_master.csv ((1620, 53))


In [10]:
# ═══ CELL 10: 3-Fold Stratified Splits (no sklearn needed) ═══
import random

def stratified_kfold(pids, labels, n_splits=3, seed=42):
    """Pure-Python stratified k-fold. No sklearn required."""
    rng = random.Random(seed)
    # Group PIDs by class
    by_class = {}
    for pid, lbl in zip(pids, labels):
        by_class.setdefault(lbl, []).append(pid)
    # Shuffle each class
    for lbl in by_class:
        rng.shuffle(by_class[lbl])
    # Assign to folds round-robin within each class
    fold_assign = {}
    for lbl, class_pids in by_class.items():
        for i, pid in enumerate(class_pids):
            fold_assign[pid] = i % n_splits
    # Build fold splits
    all_pids_arr = list(fold_assign.keys())
    folds = []
    for f in range(n_splits):
        test  = [p for p in all_pids_arr if fold_assign[p] == f]
        train = [p for p in all_pids_arr if fold_assign[p] != f]
        folds.append((train, test))
    return folds

pid_tp_counts = master.groupby('patient_id')['tp_ordinal'].count()
traj_pids = pid_tp_counts[pid_tp_counts >= 2].index.tolist()
print(f'Trajectory-capable patients (≥2 tp): {len(traj_pids)}')

pid_class = master[master['patient_id'].isin(traj_pids)].drop_duplicates('patient_id')
pids_list = pid_class['patient_id'].tolist()
labels_list = pid_class['traj_class_v2'].tolist()

splits = {}
for fold, (train_pids, test_pids) in enumerate(stratified_kfold(pids_list, labels_list, 3, 42)):
    n_val = max(1, int(len(train_pids) * 0.15))
    splits[f'fold_{fold}'] = {
        'train': train_pids[n_val:],
        'val': train_pids[:n_val],
        'test': test_pids,
    }
    print(f'  Fold {fold}: train={len(train_pids)-n_val} val={n_val} test={len(test_pids)}')

splits_path = OUTPUT_ROOT / 'brats_ptg_splits.json'
with open(splits_path, 'w') as f:
    json.dump(splits, f, indent=2)
print(f'Saved: {splits_path}')

Trajectory-capable patients (≥2 tp): 559
  Fold 0: train=317 val=55 test=187
  Fold 1: train=317 val=55 test=187
  Fold 2: train=318 val=56 test=185
Saved: /kaggle/working/ext_brats_ptg/brats_ptg_splits.json


In [11]:
# ═══ CELL 11: Summary ═══
import pandas as pd
emb_data = np.load(OUTPUT_ROOT / 'embeddings' / 'brats_ptg_embeddings.npz', allow_pickle=True)

dice_path = OUTPUT_ROOT / 'embeddings' / 'brats_ptg_dice_scores.csv'
has_dice = dice_path.exists()
if has_dice:
    dice_df = pd.read_csv(dice_path)

print('═' * 65)
print('  EXT_AB COMPLETE')
print('═' * 65)
print(f'  Embeddings: {emb_data["embeddings"].shape} ({emb_data["embeddings"].shape[1]}-D)')
print(f'  Patients:   {len(set(emb_data["patient_ids"]))}')
print(f'  Scans:      {len(emb_data["patient_ids"])}')
print()
if has_dice:
    print(f'  Segmentation Dice — MU-Glioma baseline: WT=0.904 TC=0.857 ET=0.872')
    print(f'    WT: {dice_df["dice_wt"].mean():.4f} ± {dice_df["dice_wt"].std():.4f}')
    print(f'    TC: {dice_df["dice_tc"].mean():.4f} ± {dice_df["dice_tc"].std():.4f}')
    print(f'    ET: {dice_df["dice_et"].mean():.4f} ± {dice_df["dice_et"].std():.4f}')
else:
    print('  Dice scores: skipped (using cached embeddings)')
print()
print(f'  Master CSV: {master.shape}')
print(f'  Splits: 3-fold CV on {len(traj_pids)} patients')
print()
assert emb_data['embeddings'].shape[1] == 2825
print('  ✅ 2825-D verified')
print('  ✅ Ready for EXT_C (TaViT External Validation)')

═════════════════════════════════════════════════════════════════
  EXT_AB COMPLETE
═════════════════════════════════════════════════════════════════
  Embeddings: (1620, 2825) (2825-D)
  Patients:   731
  Scans:      1620

  Segmentation Dice — MU-Glioma baseline: WT=0.904 TC=0.857 ET=0.872
    WT: 0.8475 ± 0.1083
    TC: 0.7226 ± 0.2116
    ET: 0.7167 ± 0.1954

  Master CSV: (1620, 53)
  Splits: 3-fold CV on 559 patients

  ✅ 2825-D verified
  ✅ Ready for EXT_C (TaViT External Validation)
